In [1]:
import datasets

In [2]:
dataset = datasets.load_dataset("json", data_files="../data-response_gen-i_r-e133b-0-20250516_222339.jsonl")

In [3]:
dataset

DatasetDict({
    train: Dataset({
        features: ['prompt', 'fingerprint', 'sha1', 'id', 'seed', 'concepts', 'instruction', 'parsing_result'],
        num_rows: 187
    })
})

In [4]:
dataset["train"][0]["parsing_result"][0]["response"]

"Here's how you can implement the `validate_integer_array` function:\n\n```c\nint validate_integer_array(const int* array, size_t length) {\n    if (array == NULL || length == 0) {\n        return -1;\n    }\n\n    for (size_t i = 0; i < length; ++i) {\n        if (array[i] != (int)array[i]) {\n            return -1;\n        }\n    }\n\n    for (size_t i = 0; i < length - 1; ++i) {\n        if (array[i] > array[i + 1]) {\n            return -1;\n        }\n    }\n\n    int previous = array[0];\n    for (size_t i = 1; i < length; ++i) {\n        if (array[i] == previous) {\n            return -1;\n        }\n        previous = array[i];\n    }\n\n    return 0;\n}\n```\n\nIn the first check, we verify that the array is non-null and non-empty. If either condition is violated, the function immediately returns `-1`.\n\nThe second check ensures that each element in the array is an integer. We compare each element to its own value cast to an integer, and if they differ, the element is not an

In [5]:
from tree_sitter import Language, Parser
import os


C_LANGUAGE = Language('build/lang.so', 'c')
parser = Parser()
parser.set_language(C_LANGUAGE)

def validate_code(func):
    tree = parser.parse(bytes(func, "utf8"))
    root_node = tree.root_node
    found_func = any(
        child.type == "function_definition"
        for child in root_node.children
    )

    if found_func:
        return 1
    else:
        return 0


In [6]:
success = 0
failure = 0
total_response_success = 0
seed_functions = []
skipped = 0
total_responses = 0
for _ in dataset["train"]:
    response_success = 0
    skip_responses = 0
    for response in _["parsing_result"]:
        total_responses += 1
        text = response["response"]
        if "python" in text:
            skip_responses += 1
            continue
        try:
            text = text.split("</response>\n\n<tests>")[0]
            text = text.split("```c")[1] 
            code = text.replace("```","")
        except:
            continue   
        if validate_code(code):
            response_success += 1
            validated_code = code
    total_response_success += response_success
    if response_success:
        seed_functions.append({
            "sha1": _["sha1"],
            "id": _["id"],
            "seed": validated_code,
            "concepts": _["concepts"],
            "instruction": _["instruction"] 
        })
        success += 1
    else:
        if skip_responses:
            skipped += 1
        else:
            failure += 1

    
print("-"*40)
print(f"Total functions generated: {len(dataset['train'])}")
print(f"Total responses checked: {total_responses}")
print(f"Resposnes passed syntax check: {total_response_success}")
print(f"Functions passed syntax check ✅: {success}")
print(f"Functions failed syntax check ❌: {failure}")
print(f"Functions skipped syntax check ⚠️: {skipped}")

----------------------------------------
Total functions generated: 187
Total responses checked: 1870
Resposnes passed syntax check: 1762
Functions passed syntax check ✅: 184
Functions failed syntax check ❌: 1
Functions skipped syntax check ⚠️: 2


In [7]:
seed_functions[0]

{'sha1': '895db590af78abbe4d50b4283d73927eb45613ed',
 'id': 8,
 'seed': '\nint validate_integer_array(int* array, int length) {\n    if (array == NULL || length <= 0) {\n        return -2;\n    }\n    for (int i = 0; i < length; i++) {\n        if (array[i] != i) {\n            return -1;\n        }\n    }\n    for (int i = 1; i < length; i++) {\n        if (array[i] < array[i - 1]) {\n            return -1;\n        }\n    }\n    return 0;\n}\n\n\nThe function first checks if the array is non-null and non-empty. If not, it returns `-2` to indicate an invalid array. It then iterates through the array and checks whether each element is an integer by comparing it with its index. If any element is not an integer, the function returns `-1` to indicate an invalid array. Finally, the function checks whether the array is sorted in ascending order by iterating through the array and comparing each element with the previous one. If any element is not greater than or equal to the previous one, th

In [8]:
validated_dataset = datasets.Dataset.from_list(seed_functions)
validated_dataset.to_json("final_data.jsonl", lines=True)
validated_dataset

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Dataset({
    features: ['sha1', 'id', 'seed', 'concepts', 'instruction'],
    num_rows: 184
})

In [9]:
validated_dataset["seed"][0]

'\nint validate_integer_array(int* array, int length) {\n    if (array == NULL || length <= 0) {\n        return -2;\n    }\n    for (int i = 0; i < length; i++) {\n        if (array[i] != i) {\n            return -1;\n        }\n    }\n    for (int i = 1; i < length; i++) {\n        if (array[i] < array[i - 1]) {\n            return -1;\n        }\n    }\n    return 0;\n}\n\n\nThe function first checks if the array is non-null and non-empty. If not, it returns `-2` to indicate an invalid array. It then iterates through the array and checks whether each element is an integer by comparing it with its index. If any element is not an integer, the function returns `-1` to indicate an invalid array. Finally, the function checks whether the array is sorted in ascending order by iterating through the array and comparing each element with the previous one. If any element is not greater than or equal to the previous one, the function returns `-1` to indicate an invalid array. If all checks pass